# Colab -> VS Code Remote-SSH (GPU)

Этот ноутбук автоматизирует подключение VS Code к текущему Colab runtime через SSH-туннель.

Что делает:
1. Устанавливает `openssh-server` + `cloudflared`.
2. Поднимает пользователя `colab` и SSH по ключу (без пароля).
3. Стартует tunnel `tcp://localhost:22`.
4. Печатает `host/port` и готовую команду для Windows PowerShell-скрипта проекта.

Важно: после перезапуска runtime нужно запускать ноутбук заново.

In [ ]:
# ===== 0) ВСТАВЬ СЮДА ПУБЛИЧНЫЙ SSH КЛЮЧ С ЛОКАЛЬНОЙ МАШИНЫ =====
# Получить его можно командой (в PowerShell):
# Get-Content $HOME\.ssh\id_ed25519.pub

AUTHORIZED_KEY = ""  # <-- вставь строку ssh-ed25519 AAAA... your_email

assert AUTHORIZED_KEY.strip().startswith("ssh-"), "Вставь корректный публичный SSH ключ в AUTHORIZED_KEY"

In [ ]:
# ===== 1) Установка пакетов и SSH =====
import os
import re
import time
import json
import subprocess
from pathlib import Path

def sh(cmd):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True, text=True, capture_output=True)

sh("apt-get update -y")
sh("apt-get install -y openssh-server wget curl")
sh("mkdir -p /var/run/sshd")

# cloudflared
sh("wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
sh("dpkg -i /tmp/cloudflared.deb")
print("Install complete")

In [ ]:
# ===== 2) Настройка SSH user + key-only auth =====
from pathlib import Path
import subprocess

USER = "colab"
HOME_DIR = Path(f"/home/{USER}")
SSH_DIR = HOME_DIR / ".ssh"

subprocess.run(f"id -u {USER} || useradd -m -s /bin/bash {USER}", shell=True, check=True)
SSH_DIR.mkdir(parents=True, exist_ok=True)
(SSH_DIR / "authorized_keys").write_text(AUTHORIZED_KEY.strip() + "\n", encoding="utf-8")

subprocess.run(f"chown -R {USER}:{USER} {HOME_DIR}", shell=True, check=True)
subprocess.run(f"chmod 700 {SSH_DIR}", shell=True, check=True)
subprocess.run(f"chmod 600 {SSH_DIR / 'authorized_keys'}", shell=True, check=True)

# Harden sshd config (key-only)
cfg = Path('/etc/ssh/sshd_config')
text = cfg.read_text(encoding='utf-8')
def set_or_add(content, key, value):
    import re
    pat = re.compile(rf"^\s*#?\s*{re.escape(key)}\s+.*$", re.MULTILINE)
    line = f"{key} {value}"
    if pat.search(content):
        return pat.sub(line, content)
    return content.rstrip() + "\n" + line + "\n"

for k, v in [
    ("PasswordAuthentication", "no"),
    ("PubkeyAuthentication", "yes"),
    ("PermitRootLogin", "no"),
    ("ChallengeResponseAuthentication", "no"),
    ("UsePAM", "yes"),
]:
    text = set_or_add(text, k, v)
cfg.write_text(text, encoding='utf-8')

subprocess.run("service ssh restart", shell=True, check=True)
print("SSH is ready")

In [ ]:
# ===== 3) Старт cloudflared TCP tunnel =====
import re
import time
import subprocess
from pathlib import Path

log_file = Path('/tmp/cloudflared_ssh.log')
if log_file.exists():
    log_file.unlink()

cmd = "cloudflared tunnel --url tcp://localhost:22 --no-autoupdate > /tmp/cloudflared_ssh.log 2>&1 &"
subprocess.run(cmd, shell=True, check=True)
print("Tunnel process started, waiting for endpoint...")

endpoint = None
for _ in range(60):
    time.sleep(1)
    if not log_file.exists():
        continue
    text = log_file.read_text(encoding='utf-8', errors='ignore')
    m = re.search(r"tcp://([a-zA-Z0-9.-]+):(\d+)", text)
    if m:
        endpoint = (m.group(1), int(m.group(2)))
        break

if endpoint is None:
    print(log_file.read_text(encoding='utf-8', errors='ignore')[-2000:])
    raise RuntimeError("Не удалось получить tunnel endpoint. Проверь лог выше.")

host, port = endpoint
print(f"Tunnel endpoint: {host}:{port}")

connection_info = {
    "host": host,
    "port": port,
    "user": "colab",
    "workspace_hint": "/content"
}
print(json.dumps(connection_info, indent=2))

In [ ]:
# ===== 4) Готовые команды для Windows PowerShell =====
# Запусти в локальном проекте: scripts/setup_colab_vscode_remote.ps1

ps_command = (
    f".\\scripts\\setup_colab_vscode_remote.ps1 "
    f"-TunnelHost {host} -TunnelPort {port} -Alias colab-gpu -User colab -OpenVscode"
)
print("Run locally in PowerShell:")
print(ps_command)

print("\nIf VS Code does not open automatically:")
print("1) Open VS Code")
print("2) Remote-SSH: Connect to Host...")
print("3) Select: colab-gpu")

## Troubleshooting

- Если tunnel не поднялся: перезапусти ячейку #3.
- Если VS Code не подключается: проверь, что в локальном `~/.ssh/config` записан актуальный `host/port`.
- После restart/disconnect Colab endpoint меняется — скрипт на локальной машине нужно запускать заново.